In [1]:
import importlib.util
import sys
missing = [package for module, package in [
    ('numpy', 'numpy'), ('pandas', 'pandas'), ('sklearn', 'scikit-learn')
] if importlib.util.find_spec(module) is None]
if missing:
    get_ipython().run_line_magic('pip', 'install ' + ' '.join(missing))

from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from IPython.display import display
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)

Python: 3.13.0
pandas: 3.0.6 | scikit-learn: 1.9.1


 1. Load the data and make the classroom split


In [ ]:
candidates = [Path.cwd(), Path.cwd().parent,
              Path('/Users/mainguyenhong/Documents/mai/310.Exercise1')]
ROOT = next((p for p in candidates if (p / 'DSDA310' / 'adult.data').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open IC_9.ipynb in 310.Exercise1 with DSDA310/adult.data present.')

COLS = ['age', 'workclass', 'fnlwgt', 'education', 'education_num',
        'marital_status', 'occupation', 'relationship', 'race', 'sex',
        'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']
df = pd.read_csv(ROOT / 'DSDA310' / 'adult.data', header=None,
                 names=COLS, skipinitialspace=True)
y = (df.income == '>50K').values
train = (df.fnlwgt % 10 < 8).values
te = df[~train].reset_index(drop=True)
yte = y[~train]
NUM = ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
assert set(df.income.unique()) == {'<=50K', '>50K'}
assert len(df) == 32561 and train.sum() == 26193
print('Data:', ROOT / 'DSDA310' / 'adult.data')
print(f'Training rows: {train.sum():,}; held-out rows: {(~train).sum():,}')

In [3]:
classroom_tables = {}
classroom_predictions = {}

def run(cat, label):
    X = pd.get_dummies(df[NUM + cat], columns=cat)
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                random_state=0, n_jobs=-1).fit(X[train], y[train])
    pred = rf.predict_proba(X[~train])[:, 1] >= 0.5
    print(f'\n--- {label} ({X.shape[1]} columns)')
    rows = []
    rec = {}
    for g in ['Male', 'Female']:
        m = (te.sex == g).values
        rec[g] = recall_score(yte[m], pred[m])
        rows.append({'Group': g, 'n': int(m.sum()),
                     'Base rate': yte[m].mean(), 'Selected': pred[m].mean(),
                     'Precision': precision_score(yte[m], pred[m]),
                     'Recall': rec[g]})
    table = pd.DataFrame(rows)
    classroom_tables[label] = table
    classroom_predictions[label] = pred
    display(table.round(3))
    gap = rec['Male'] - rec['Female']
    print(f'Recall gap: {gap:.3f} ({gap * 100:.2f} percentage points)')
    return gap

In [12]:
a = run(['workclass', 'marital_status', 'occupation', 'race', 'sex'],
        'Model A')

NameError: name 'run' is not defined

In [16]:
b = run(['workclass', 'marital_status', 'occupation'], 'Model B')
print(f'\nThe gap went {a:.3f} -> {b:.3f} ({b - a:+.3f})')
print(f'Change in the recall gap: {(b - a) * 100:+.2f} percentage points')

NameError: name 'run' is not defined

In [15]:
proxy = pd.crosstab(te.sex, te.marital_status, normalize='index')
display(proxy[['Married-civ-spouse']].rename(columns={'Married-civ-spouse': 'Share married, civilian spouse'}).round(3))

NameError: name 'te' is not defined